In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from collections import Counter
import os

# Caminho para a Silver (saída do write_silver.py)
silver_path = "../../data/silver/cision_news_20251110.parquet"

if not os.path.exists(silver_path):
    raise FileNotFoundError(f" Ficheiro Silver não encontrado em {silver_path}")

df = pd.read_parquet(silver_path)
print(f"1Silver carregada com {len(df)} notícias únicas")
df.head(3)


c:\Users\inesc\Downloads\ISCTE\Planapp\Planapp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1Silver carregada com 43665 notícias únicas


,id,titulo,noticia_raw,noticia_norm,link,data_publicacao,tipo_meio,autores,valor_publicitario,guid,categoria,grupo_id,is_lider,processed_at
0,116477697,Isabel Ferreira reafirma compromisso com Braga...,A candidata à Presidência da Câmara Municipal ...,a candidata à presidência da câmara municipal ...,No link available,2025-04-03 00:00:00+00:00,Imprensa,sem autor,139.4,http://pt.cision.com/cp2013/clippingdetails.as...,None,0,True,2025-11-10T15:53:56Z
1,116477659,Legislativas 25 - Júlia Rodrigues escolhida pa...,"A presidente da Câmara de Mirandela, Júlia Rod...",a presidente da câmara de mirandela júlia rodr...,No link available,2025-04-03 00:00:00+00:00,Imprensa,"Fernando Pires, António Gonçalves Rodrigues",267.2,http://pt.cision.com/cp2013/clippingdetails.as...,None,1,True,2025-11-10T15:53:56Z
2,116456100,Alargamento dos passeios na envolvente à Casa-...,"Arrancou na passada segunda-feira, o alargamen...",arrancou na passada segunda feira o alargament...,No link available,2025-04-02 00:00:00+00:00,Imprensa,sem autor,100.0,http://pt.cision.com/cp2013/clippingdetails.as...,None,2,True,2025-11-10T15:53:56Z


In [2]:
from sentence_transformers import SentenceTransformer
import time

# Criar amostra 10%
df_sample = df.sample(frac=0.10, random_state=42).reset_index(drop=True)

print(f"Amostra com {len(df_sample)} notícias")

# Carregar modelo
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Gerar embeddings
start = time.time()
embeddings = model.encode(
    df_sample["noticia_norm"].astype(str).tolist(),
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)
print(f"Embeddings criados: {embeddings.shape}")
print(f"Tempo total: {(time.time()-start)/60:.1f} minutos")




Amostra com 4366 notícias


Batches: 100%|██████████| 273/273 [02:31<00:00,  1.80it/s]

Embeddings criados: (4366, 384)
Tempo total: 2.5 minutos


In [6]:
import umap

umap_reducer = umap.UMAP(n_components=10, random_state=42)
embeddings_reduced = umap_reducer.fit_transform(embeddings)

print(f"Dimensão original: {embeddings.shape[1]} → Reduzida: {embeddings_reduced.shape[1]}")


c:\Users\inesc\Downloads\ISCTE\Planapp\Planapp\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Dimensão original: 384 → Reduzida: 10


In [7]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

Ks = [16, 18, 22]
scores = []

for k in Ks:
    labels = KMeans(n_clusters=k, random_state=42).fit_predict(embeddings_reduced)
    score = silhouette_score(embeddings_reduced, labels)
    print(f"k={k}: Silhouette={score:.3f}")
    scores.append((k, score))

best_k, best_score = max(scores, key=lambda x:x[1])
print(f"\nMelhor k = {best_k}, Silhouette = {best_score:.3f}")

df_sample["cluster_id"] = KMeans(n_clusters=best_k, random_state=42).fit_predict(embeddings_reduced)


k=16: Silhouette=0.355
k=18: Silhouette=0.333
k=22: Silhouette=0.332

Melhor k = 16, Silhouette = 0.355


In [8]:
from sklearn.cluster import KMeans
from collections import Counter
import nltk
from nltk.corpus import stopwords

k_escolhido = 18 

# Treinar K-Means final
kmeans_final = KMeans(n_clusters=k_escolhido, random_state=42, n_init=10)
df_sample["cluster_id"] = kmeans_final.fit_predict(embeddings_reduced)

# Preparar stopwords PT
nltk.download('stopwords')
stop_words = set(stopwords.words("portuguese"))

print("\nPalavras mais comuns por cluster:")

for c in sorted(df_sample["cluster_id"].unique()):
    palavras = " ".join(df_sample[df_sample.cluster_id == c]["noticia_norm"])
    tokens = [p for p in palavras.split() if p not in stop_words and len(p) > 3]
    mais_comuns = Counter(tokens).most_common(15)

    print(f"\nCluster {c}:")
    print(", ".join([p[0] for p in mais_comuns]))





[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\inesc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



Palavras mais comuns por cluster:

Cluster 0:
governo, presidente, república, sobre, ministro, assembleia, estado, ainda, regional, euros, primeiro, portugal, chega, madeira, municipal

Cluster 1:
euros, saúde, social, investimento, milhões, câmara, escola, presidente, projeto, centro, municipal, ainda, obra, plano, apoio

Cluster 2:
saúde, médicos, cuidados, hospital, profissionais, serviço, anos, ainda, trabalho, médico, nacional, medicina, presidente, pessoas, sobre

Cluster 3:
europeia, comissão, sobre, europeu, união, presidente, europa, milhões, euros, estados, ainda, política, trump, tarifas, portugal

Cluster 4:
habitação, câmara, construção, euros, municipal, casas, governo, ainda, anos, presidente, milhões, investimento, habitações, concelho, reabilitação

Cluster 5:
empresas, inovação, digital, indústria, formação, trabalho, portugal, setor, ensino, emprego, anos, desenvolvimento, ainda, mercado, euros

Cluster 6:
câmara, euros, obra, construção, municipal, presidente, inve

In [ ]:
# TF-IDF para extrair termos importantes por cluster

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

print("\n🔍 A calcular TF-IDF por cluster...\n")

# stopwords tem de ser LISTA para o TF-IDF
stop_words = list(stopwords.words("portuguese"))

vectorizer = TfidfVectorizer(
    max_features=3000,
    stop_words=stop_words,
    ngram_range=(1,2)   # inclui bigramas
)

for c in sorted(df_sample["cluster_id"].unique()):
    textos = df_sample[df_sample.cluster_id == c]["noticia_norm"].tolist()

    if len(textos) < 3:
        print(f"\nCluster {c}: (poucos textos)")
        continue

    # TF-IDF para o cluster
    tfidf_matrix = vectorizer.fit_transform(textos)
    scores = tfidf_matrix.mean(axis=0).A1
    termos = vectorizer.get_feature_names_out()

    # top termos
    top_idx = scores.argsort()[::-1][:15]
    top_termos = [termos[i] for i in top_idx]

    print(f"\n Cluster {c} — termos TF-IDF mais relevantes:")
    print(", ".join(top_termos))



🔍 A calcular TF-IDF por cluster...


 Cluster 0 — termos TF-IDF mais relevantes:
governo, ministro, ps, presidente, luís, chega, república, declarações, primeiro, parlamento, estado, josé, sobre, primeiro ministro, vai

 Cluster 1 — termos TF-IDF mais relevantes:
saúde, euros, social, escola, investimento, milhões, câmara, centro, projeto, obra, unidade, vai, presidente, ano, alunos

 Cluster 2 — termos TF-IDF mais relevantes:
saúde, médicos, hospital, sns, ministra, serviço, profissionais, presidente, ordem, médico, cuidados, nacional, enfermeiros, declarações, martins

 Cluster 3 — termos TF-IDF mais relevantes:
europeia, comissão, presidente, trump, comissão europeia, tarifas, europeu, união, sobre, ue, estados, milhões, união europeia, europa, euros

 Cluster 4 — termos TF-IDF mais relevantes:
habitação, câmara, euros, construção, municipal, casas, governo, milhões, habitações, reabilitação, arrendamento, fogos, milhões euros, mil, investimento

 Cluster 5 — termos TF-IDF mais rel

In [33]:
print("------ CLUSTER 0 ------")
print("\n".join(df_sample[df_sample.cluster_id == 0]["noticia_norm"].head(10)))

print("\n------ CLUSTER 3 ------")
print("\n".join(df_sample[df_sample.cluster_id == 3]["noticia_norm"].head(10)))



------ CLUSTER 0 ------
rui rocha despediu se da liderança da il na convenção em alcobaça com recados para dentro e fora admitiu que não foram cumpridos os objetivos e apelou ao partido para olhar para o futuro no arranque da x convenção da iniciativa liberal il rui rocha subiu ao palco para um discurso enquanto presidente cessante e aquilo que seria um balanço do seu mandato revelou se um claro ajuste de contas com muitas farpas aos críticos à mistura e recados para a sucessora o ex líder dos liberais garantiu estar de consciência tranquila e deixa o cargo com o mesmo desprendimento dos seus antecessores sublinhando que em dois anos e meio a il passou a estar presente em todos os parlamentos e tem uma visão estratégica e autonomia política e financeira para o futuro cabe a mariana leitão provar agora que o partido vale mais do que 5% dos votos houve vozes revelantes na ll que comentaram resultados mas a decisão foi minha e essas vozes foram absolutamente irrelevantes na minha tomada d

In [38]:
print("\n------ CLUSTER 1 ------")
print("\n".join(df_sample[df_sample.cluster_id == 1]["noticia_norm"].head(10)))

print("\n------ CLUSTER 2 ------")
print("\n".join(df_sample[df_sample.cluster_id == 2]["noticia_norm"].head(10)))



------ CLUSTER 1 ------
a casa dos crivos situada na rua de s marcos em braga vai entrar em obras de reabilitação e restauro no próximo mês de julho a notícia foi confirmada pelo presidente da câmara de braga que realçou estar tudo praticamente pronto para que os trabalhos avancem «já tinha sido lançado o concurso tinham sido cumpridas todas as formalidades administrativas necessárias e neste momento já estão em curso os preparativos para o arranque da obra que vai de facto iniciar se no início do mês de julho» disse ricardo rio ao diário do minho ainda segundo o autarca o concurso aponta para que as obras decorram num prazo de 150 dias ou seja cinco meses pelo que tudo deverá estar concluído no próximo mês de de zembro o valor orçamentado para os trabalhos é de cerca de 400 mil euros «são várias as intervenções que vão ser feitas quer do ponto de vista da salvaguarda patrimonial quer também da melhoria das condições de funcionalidade do espaço e portanto eu diria que até ao fni al do

In [39]:
print("\n------ CLUSTER 4 ------")
print("\n".join(df_sample[df_sample.cluster_id == 4]["noticia_norm"].head(10)))


------ CLUSTER 4 ------
“estar de banco” hoje é muito mais do que cumprir um turno é sentar se metaforicamente no mesmo banco onde começou a ciência dos nossos hospitais poucos lisboetas imaginam que uma expressão tão corriqueira e viva como “estou de banco” dita por médicos e enfermeiros a anunciar o início do seu turno carrega séculos de história dentro dela é uma dessas heranças silenciosas que atravessam o tempo nascidas nas salas austeras do real hospital de todos os santos onde se misturavam o saber o cheiro dos unguentos e o rumor dos sofredores que enchiam os claustros da cidade nos tempos em que a medicina era ainda mais arte do que ciência o “banco dos médicos” era um lugar físico real de madeira gasta e de trabalho constante era ali que se sentavam os cirurgiões e os físicos do rei debruçados sobre os frascos de vidro translúcido que guardavam amostras de urina as célebres matulas erguidas contra a luz para avaliar cor transparência e espuma o “banco” era a bancada onde se 

In [10]:
from dotenv import load_dotenv
import os
from openai import OpenAI

# Carregar variáveis do ficheiro .env
load_dotenv()

# Ler a chave a partir do .env
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

dicionario_cluster_para_nome = {}

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Diz apenas 'Ligação OK'"}]
)

print(response.choices[0].message.content)



Ligação OK


In [11]:
from collections import Counter
import nltk
from nltk.corpus import stopwords
import pandas as pd # Necessário para o .join na extração

# --- PARTE 1: EXTRAÇÃO DE PALAVRAS-CHAVE (TF/IDF SIMPLIFICADO) ---

# Garantir stopwords em português
try:
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    nltk.download('stopwords')
stop_words = set(stopwords.words('portuguese'))


cluster_palavras = {}

for c in sorted(df_sample["cluster_id"].unique()):
    # Junta todo o texto de um cluster
    palavras = " ".join(df_sample[df_sample.cluster_id == c]["noticia_norm"].astype(str))

    # Tokens filtrados (remove stopwords e pontuação)
    tokens = [
        p.lower()
        for p in palavras.split()
        if p.lower() not in stop_words and len(p) > 2
    ]

    # Guarda as 15 palavras mais comuns
    mais_comuns = [p[0] for p in Counter(tokens).most_common(15)]
    cluster_palavras[c] = mais_comuns

print("✔ Palavras mais frequentes extraídas por cluster.\n")


# --- PARTE 2: ROTULAGEM DOS CLUSTERS (USANDO GPT-4o-mini E 19 FEW-SHOTS) ---

# Lista RÍGIDA das 19 categorias de política pública (A sua taxonomia)
areas_oficiais = """
Agricultura e Floresta; Administração Pública; Ambiente e Clima; Cultura; Defesa;
Desporto; Economia; Educação e Formação; Energia; Habitação; Impostos e taxas;
I&D e Inovação; Justiça; Mar e Pescas; Proteção Social; Saúde; Segurança;
Trabalho; Transportes e Mobilidade
"""

dicionario_cluster_para_nome = {}
# ⚠️ Assume-se que 'client' (OpenAI) está definido e ativo.
print("A rotular os clusters (GPT-4o-mini)...")

# Construção dos 19 exemplos Few-Shot (o prompt extenso)
few_shot_examples = """
Exemplos de Classificação (Obrigatório):

Palavras: hospitais, médicos, SNS, urgências, cuidados de saúde, enfermeiros, vacinação
Categoria: Saúde

Palavras: futebol, campeonato, liga, equipa, golo, estádio, atleta, modalidade, jogos
Categoria: Desporto

Palavras: agricultura, floresta, gado, vinha, produção agrícola, incêndios rurais, campo
Categoria: Agricultura e Floresta

Palavras: tribunais, juiz, advogados, julgamento, processo, procurador, lei, justiça
Categoria: Justiça

Palavras: inflação, investimento, PIB, mercado, exportação, bancos, impostos, finanças, economia
Categoria: Economia

Palavras: câmara municipal, autarquia, serviços públicos, concurso, funcionários, estado
Categoria: Administração Pública

Palavras: clima, aquecimento global, poluição, sustentabilidade, emissões, reciclagem
Categoria: Ambiente e Clima

Palavras: museu, teatro, cinema, artes, património, espetáculo, festival, cultura
Categoria: Cultura

Palavras: defesa nacional, forças armadas, exército, marinha, aeronáutica, missão militar
Categoria: Defesa

Palavras: escola, ensino, alunos, professores, universidade, formação profissional, aulas
Categoria: Educação e Formação

Palavras: energia, petróleo, gás, eletricidade, renovável, solar, eólica, transmissão
Categoria: Energia

Palavras: habitação, renda, imobiliário, arrendamento, casa, obras, construção, crédito
Categoria: Habitação

Palavras: impostos, IVA, IRS, taxas, contribuições, autoridade tributária, fiscal
Categoria: Impostos e taxas

Palavras: investigação, inovação, ciência, startup, tecnologia, laboratório, descoberta
Categoria: I&D e Inovação

Palavras: mar, pescas, oceano, porto, pescadores, costa, navio, recursos marinhos
Categoria: Mar e Pescas

Palavras: segurança social, pensões, reformas, apoios sociais, subsídio, idosos
Categoria: Proteção Social

Palavras: segurança, PSP, GNR, polícia, crime, violência, roubo, investigação, assalto
Categoria: Segurança

Palavras: trabalho, emprego, salário, desemprego, contrato, empresa, horário, sindicato
Categoria: Trabalho

Palavras: transportes, mobilidade, carro, comboio, avião, autocarro, metro, estrada, aeroporto
Categoria: Transportes e Mobilidade
"""

for c, palavras in cluster_palavras.items():

    prompt = f"""
Classifica o seguinte grupo de palavras-chave numa categoria de políticas públicas portuguesas.

Regras de Classificação:
- A resposta TEM de ser UMA das categorias listadas em "Categorias disponíveis".
- Escolhe apenas UMA categoria, a mais representativa.
- Não cries categorias novas.
- Responde APENAS com o nome da categoria.

{few_shot_examples}

Categorias disponíveis:
{areas_oficiais}

Palavras do cluster:
{', '.join(palavras)}

Categoria:
"""
    # ... Resto da chamada API para o GPT (client.chat.completions.create)
    resposta = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    categoria_nome = resposta.choices[0].message.content.strip()
    dicionario_cluster_para_nome[c] = categoria_nome

    print(f"Cluster {c} ➜ {categoria_nome}")

print("\n✔ Nomes de categorias gerados com sucesso!")


   

✔ Palavras mais frequentes extraídas por cluster.

A rotular os clusters (GPT-4o-mini)...
Cluster 0 ➜ Administração Pública
Cluster 1 ➜ Administração Pública
Cluster 2 ➜ Saúde
Cluster 3 ➜ Economia
Cluster 4 ➜ Habitação
Cluster 5 ➜ I&D e Inovação
Cluster 6 ➜ Administração Pública
Cluster 7 ➜ Administração Pública
Cluster 8 ➜ Administração Pública
Cluster 9 ➜ Administração Pública
Cluster 10 ➜ Trabalho
Cluster 11 ➜ Administração Pública
Cluster 12 ➜ Administração Pública
Cluster 13 ➜ Administração Pública
Cluster 14 ➜ Cultura
Cluster 15 ➜ Economia
Cluster 16 ➜ Administração Pública
Cluster 17 ➜ Economia

✔ Nomes de categorias gerados com sucesso!


In [1]:
import pandas as pd

df_gold = pd.read_csv("../../data_sample/golden_set_10_por_categoria.csv")


In [2]:
# confirmar distribuição
df_gold["Categoria"].value_counts()


Categoria
Saúde                   10
Desporto                10
Agricultura_Floresta    10
Justiça                 10
Economia                10
Admin_Publica           10
Ambiente_Clima          10
Cultura                 10
Defesa                  10
Educação                10
Energia                 10
Habitação               10
Impostos                10
ID_Inovação             10
Mar_Pescas              10
Proteção_Social         10
Segurança               10
Trabalho                10
Transportes             10
Name: count, dtype: int64

In [3]:
df_gold[["Categoria", "id", "titulo"]]


,Categoria,id,titulo
0,Saúde,118805777,Médicos não trocam sossego do interior do país...
1,Saúde,119125198,Autoras de ´Uma Aventura´ explicam o SNS aos j...
2,Saúde,119834711,Governo mexe no diploma das urgências regionai...
3,Saúde,118032180,Médicos de Família e a Complementaridade do SN...
4,Saúde,118669104,Índice Saúde Sustentável 2024/25 - Portugueses...
...,...,...,...
185,Transportes,119231006,Rail Baltica - Uma nova linha férrea a pensar ...
186,Transportes,117098456,"Viseu quer comboio em ""velocidade elevada"" e ""..."
187,Transportes,117417867,Mealhada celebra feriado com homenagens - Entr...
188,Transportes,116735331,Penela uma Vila de experiências que pode torna...
